In [1]:
from tnwater import load_gps, load_water_quality, merge_water_quality_with_gps, filter_ids

nut_df = load_water_quality("../data/wq_data_for_tennessee.csv")
gps = load_gps("../data/dam_distances.csv")
merged_df = merge_water_quality_with_gps(nut_df=nut_df, gps_df=gps)

# something about the merging is broken

C:\Users\Spencer Womble\OneDrive\TN_Tech2\projects\reservoir_work\analysis_files\project_folder\tnwater\pipeline.py:26: DtypeWarning: Columns (0: End Date, 1: Measure Qualifier Code, 2: Measure Qualifier Description, 3: Quantitation Limit Unit Code, 4: Activity Depth Unit Code, 5: Analytical Method Identifier, 6: Result Speciation, 7: Result Value Type, 8: Result Detection Condition, 9: Result Status, 10: Quantitation Limit Type, 11: Quantitation Limit Speciation, 12: Sample Collection Method Identifier, 13: Sample Collection Method, 14: Sample Collection Method Description, 15: Sample Collection Method Context Code, 16: Sample Collection Method Context, 17: Sample Collection Equipment, 18: Start Time Zone Code, 19: End Time Zone Code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [2]:
# get ride of duplicate observations by random sampling (phosphorus seems to have a lot of duplicates)
group_keys = ['date_time', 'site', 'characteristic', 'sample_fraction']

deduped = (
    merged_df.sample(frac=1, random_state=42)              # shuffle all rows - important to do first as we shuffle and then keep the first row for each duplicate
      .drop_duplicates(subset=group_keys, keep='first')
      .sort_values('date_time')
      .reset_index(drop=True)
)

In [3]:
# filter down to the location IDs that have good data coverage. The default contains the location IDs, but you can manually specify them if you want to.gps

filtered_df = filter_ids(df=deduped)

In [4]:
# check if all nutrient/tss values are in mg/L
mask = (filtered_df["measure_unit_code"] == "mg/l") & (filtered_df["sample_fraction"].notna())
filtered_df[mask]["measure_unit_code"].unique()



<StringArray>
['mg/l']
Length: 1, dtype: str

In [ ]:
from tnwater import pivot_wider

# pivot data to wide format for plotting and modeling
filtered_df_wide = pivot_wider(filtered_df)


In [ ]:
from tnwater import clean_column_names
# clean column names for the new nutrient variable columns

filtered_df_wide = clean_column_names(filtered_df_wide)


In [8]:
# extract year and month from date_time
from tnwater import decimal_date

filtered_df_wide['year'] = filtered_df_wide['date_time'].dt.year
filtered_df_wide['month'] = filtered_df_wide['date_time'].dt.month
filtered_df_wide['decimal_date'] = decimal_date(filtered_df_wide['date_time'])


In [9]:
# write filtered wide data frame to new csv

filtered_df_wide.to_csv('../data/cleaned_data.csv', index=False)